# LookTwice — 01 Preprocessing (Kaggle)

1. **Add data**: dataset DFD (`sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset`).
2. **Settings → Internet: ON** (untuk `git clone` / `pip` / download YuNet).
3. Isi `REPO_URL` (repo GitHub) **atau** upload folder `look-twice` sebagai Kaggle Dataset lalu isi `REPO_DATASET`.
4. Setelah selesai: *Save Version*, lalu jadikan output `/kaggle/working/data` sebagai Kaggle Dataset baru (`looktwice-faces`).

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/garrentanavaro/look-twice.git'
REPO_DATASET = ''
WORK = '/kaggle/working/look-twice'
if not os.path.exists(WORK):
    if REPO_URL:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, WORK], check=True)
    else:
        subprocess.run(['cp', '-r', REPO_DATASET, WORK], check=True)
os.chdir(WORK)
!pip install -q -e .
!python -c "import cv2; print('opencv', cv2.__version__, hasattr(cv2, 'FaceDetectorYN'))"

## Cari folder dataset

In [ ]:
!ls /kaggle/input
DFD_ROOT = '/kaggle/input/deep-fake-detection-dfd-entire-original-dataset'
!ls "$DFD_ROOT"

## 1. Inspect: cek format nama file & jumlah aktor
Pastikan `unparsed` dan `unlabeled` = 0, jumlah aktor sekitar 28, dan contoh nama fake berbentuk `01_02__scene__ID`.

In [ ]:
!python -m looktwice.preprocess inspect --root "$DFD_ROOT" --out /kaggle/working/data/meta

## 2. Split per aktor
Lihat dulu seed terbaik, lalu bandingkan policy `strict` vs `target` dari angka yang dicetak.

In [ ]:
!python -m looktwice.preprocess split --out /kaggle/working/data/meta --search-seeds 500

In [ ]:
SEED = 42
!python -m looktwice.preprocess split --out /kaggle/working/data/meta --seed $SEED --policy strict

## 3. Smoke test ekstraksi (5 video per kelas) + lihat hasil crop

In [ ]:
!python -m looktwice.preprocess extract --root "$DFD_ROOT" --meta /kaggle/working/data/meta --out /kaggle/working/smoke --limit 5

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, cv2
idx = pd.read_csv('/kaggle/working/smoke/crops_index.csv')
sample = idx.groupby('label').head(8)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(cv2.cvtColor(cv2.imread('/kaggle/working/smoke/' + r.path), cv2.COLOR_BGR2RGB))
    ax.set_title(('fake' if r.label else 'real') + f' {r.video_id[:10]}', fontsize=7); ax.axis('off')
plt.tight_layout()

## 4. Full run
Kalau crop di atas sudah benar (wajah di tengah, ada batas rambut/dagu).

In [ ]:
!rm -rf /kaggle/working/smoke
!python -m looktwice.preprocess extract --root "$DFD_ROOT" --meta /kaggle/working/data/meta --out /kaggle/working/data/faces --workers 4

In [ ]:
idx = pd.read_csv('/kaggle/working/data/faces/crops_index.csv')
print(idx.groupby(['split', 'label']).size().unstack())
fails = pd.read_csv('/kaggle/working/data/faces/failures.csv')
print('failed frames:', len(fails)); fails.head()